<a href="https://colab.research.google.com/github/ZiyanNifail/LearningProjectPython/blob/main/Project_Sentiment_Stock_Analysis_using_spaCy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Sentiment Analysis & Stock Price Prediction
### Using spaCy NER, TextBlob Sentiment, and Machine Learning

This notebook explores a core question in quantitative finance:

> **Does the *tone* of news coverage about a company predict how its stock moves the next trading session?**

We examine two companies as case studies:

| Company | Exchange | Ticker | Currency |
|---|---|---|---|
| Apple Inc. | NASDAQ (US) | `AAPL` | USD |
| Samsung Electronics | KRX (Korea) | `005930.KS` | KRW |

---

### ⚠️ The Most Important Lesson — Read Before Anything Else

Financial markets are highly **efficient**. Algorithmic trading systems read and react to news in milliseconds. By the time a human analyst reads a headline, the signal has already been reflected in the price. Any prediction model we build here is expected to show **weak, noisy results** — that is the correct outcome. Understanding *why* models barely beat random chance is a deep and practical lesson in financial data science.

---

### 🔁 The Full Pipeline

```
[1] Fetch live headlines ──── yfinance .news  (no API key required)
         │
         ▼
[2] Company detection ──────── spaCy NER  +  PhraseMatcher (alias matching)
         │
         ▼
[3] Sentiment scoring ─────── spacytextblob → polarity  (−1 to +1)
                                            → subjectivity (0 to 1)
         │
         ▼
[4] Next-session returns ──── yfinance .download()  +  timezone alignment
         │
         ▼
[5] Labelled feature matrix── polarity, subjectivity, token_count, company
         │
         ▼
[6] Model comparison ──────── Logistic Regression  |  Random Forest  |  Gradient Boosting
         │
         ▼
[7] Honest evaluation ─────── Recall · F1 · ROC-AUC   (not raw accuracy)
```

In [ ]:
# NER >> Named Entity Recognition

# Apple just broke all time record

# NER will be able to identify Apple as an organization

# iPhone 18 is about to release in Sept

# NER will process the iPhone 18 as Apple

---
## 📦 Step 0 — Install Dependencies

We install everything in a **single cell** at the top. This is a Colab best practice — easy to re-run from scratch after a runtime reset.

| Package | Purpose |
|---|---|
| `spacy` | Core NLP engine — tokenization, NER, pipeline management |
| `spacytextblob` | Adds TextBlob sentiment as a native spaCy pipeline component |
| `en_core_web_sm` | Pre-trained small English NLP model from spaCy |
| `yfinance` | Downloads stock news and price history from Yahoo Finance |
| `scikit-learn` | ML — model training, preprocessing, evaluation metrics |
| `plotly` | Interactive visualizations |
| `pytz` / `tzdata` | Timezone conversion for Seoul ↔ New York alignment |

> **Colab Note:** After this cell finishes, you may see a **"Restart Runtime"** prompt. Click it before running any other cells. This ensures newly installed packages are picked up correctly.

In [ ]:
!pip install spacy spacytextblob yfinance -q

In [ ]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 94.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
from spacy.matcher import PhraseMatcher
from spacytextblob.spacytextblob import SpacyTextBlob

import yfinance as yf
import pandas as pd
import numpy as np

---
## 🔍 Section 1 — Named Entity Recognition (NER) with spaCy

### What is NLP?

**Natural Language Processing (NLP)** is a field of AI that teaches computers to understand human language — identifying grammatical structure, meaning, and named real-world objects.

### What is Named Entity Recognition?

**NER** is the task of automatically detecting and classifying named objects in text: people, organisations, places, dates, money amounts, and more.

**Example:**
```
"Apple announced a $3 billion acquisition in Cupertino on Tuesday."
 ─────           ──────────────────                     ───────
  ORG                  MONEY                              DATE
```

spaCy uses a neural model trained on millions of annotated documents to predict entity labels automatically.

---

### 🛠️ Functions Used in This Cell

#### `spacy.load(name)`
Loads a pre-trained NLP pipeline from disk.

| Parameter | Type | Description |
|---|---|---|
| `name` | `str` | The model package name or file path to load. We use `"en_core_web_sm"` — the small English model downloaded in Step 0. |

`en_core_web_sm` bundles four components that run in sequence:
1. **Tokenizer** — splits raw text into individual words and punctuation tokens
2. **Tagger** — assigns part-of-speech labels (noun, verb, adjective…)
3. **Parser** — identifies grammatical relationships (subject, object…)
4. **NER** — detects and classifies named entities

> 💡 `sm` = small (~12 MB, fast). `md` = medium (more accurate). `lg` = large (highest accuracy, slowest). For classroom use `sm` is the right trade-off.

#### `nlp(text)`
Processes a string through the full NLP pipeline and returns a `Doc` object.

| Parameter | Type | Description |
|---|---|---|
| `text` | `str` | The raw string to process — a single headline in our case |

Returns a `Doc` object: a rich container holding tokens, part-of-speech tags, parse tree, detected entities, and (after we add it) sentiment scores.

#### `doc.ents`
A tuple of `Span` objects, one per detected named entity in the `Doc`.

Each `Span` exposes:
- `.text` — the raw matched string (e.g., `"Apple"`)
- `.label_` — the entity type string (e.g., `"ORG"`, `"PERSON"`, `"GPE"`, `"DATE"`, `"MONEY"`)
- `.start_char` / `.end_char` — character positions in the original text

In [ ]:
# Loading the small english NLP model

nlp = spacy.load("en_core_web_sm")

print(f"Named of pipelines in NLP: {nlp.pipe_names}")
print(f"Language of NLP : {nlp.lang}")

Named of pipelines in NLP: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
Language of NLP : en


In [ ]:
sample_headlines = [
    "A company reports record revenue of $90 billion, iPhone sales beat estimates",
    "Samsung unveils new Galaxy AI chip, shares rise on Korean exchange",
    "Tim Cook speaks at Cupertino event as Apple stock hits all-time high",
    "Fed raises interest rates — technology stocks tumble globally",
]

In [ ]:
for headlines in sample_headlines:
  doc = nlp(headlines)
  print(f"Headlines: {headlines}")
  if doc.ents:
    for ent in doc.ents:
      print(f"Label: {ent.label_} Text: {ent.text}")
    else:
      print('None')

Headlines: A company reports record revenue of $90 billion, iPhone sales beat estimates
Label: MONEY Text: $90 billion
None
Headlines: Samsung unveils new Galaxy AI chip, shares rise on Korean exchange
Label: ORG Text: Samsung
Label: PERSON Text: Galaxy AI
Label: NORP Text: Korean
None
Headlines: Tim Cook speaks at Cupertino event as Apple stock hits all-time high
Label: PERSON Text: Tim Cook
Label: GPE Text: Cupertino
Label: ORG Text: Apple
None
Headlines: Fed raises interest rates — technology stocks tumble globally
Label: ORG Text: Fed
None


---
## 🎯 Section 2 — Catching Company Aliases with PhraseMatcher

### Why NER Alone Is Not Enough

spaCy's neural NER is powerful, but it has systematic blind spots for financial text:

| Problem | Example text | NER result |
|---|---|---|
| Ticker symbols | `"AAPL rose 3% today"` | Likely misses `AAPL` as ORG |
| Common word as name | `"Apple beats forecasts"` | May classify as PRODUCT, not ORG |
| Informal references | `"the Cupertino giant"` | NER misses entirely |
| Brand products as proxies | `"iPhone demand surges"` | Not tagged as Apple |

**PhraseMatcher** is a rule-based fallback: we explicitly define every alias we care about, and the matcher finds all occurrences — fast and deterministically.

### The Combined Strategy

```
NER  ─────────  catches well-known entities the neural model recognises
   +
PhraseMatcher ── catches tickers, informal references, product names we define
   =
Complete, reliable company detection
```

---

### 🛠️ Functions Used in This Cell

#### `PhraseMatcher(vocab, attr)`
Creates a phrase-matching engine attached to a spaCy vocabulary.

| Parameter | Type | Description |
|---|---|---|
| `vocab` | `spacy.vocab.Vocab` | The vocabulary of the loaded `nlp` model. Patterns must be created using the same vocabulary. Pass `nlp.vocab`. |
| `attr` | `str` | Which token attribute to compare during matching. `"LOWER"` = case-insensitive matching, so `"apple"`, `"Apple"`, and `"APPLE"` all trigger the same pattern. Default `"ORTH"` = exact case match. |

#### `nlp.make_doc(text)`
Creates a lightweight `Doc` from a string without running the full pipeline. Used to convert alias strings into matchable pattern objects efficiently.

| Parameter | Type | Description |
|---|---|---|
| `text` | `str` | The alias phrase to convert into a pattern Doc |

> ⚠️ Must use the **same** `nlp` object as the matcher — vocabulary must match, or matches will silently fail.

#### `matcher.add(key, patterns)`
Registers a group of phrase patterns under a label.

| Parameter | Type | Description |
|---|---|---|
| `key` | `str` | The label returned when any pattern in this group matches. We use `"APPLE"` and `"SAMSUNG"`. |
| `patterns` | `list[Doc]` | A list of Doc objects (from `nlp.make_doc()`). Each Doc is one alias phrase to watch for. |

#### `matcher(doc)`
Runs all registered patterns over a Doc and returns every match found.

| Parameter | Type | Description |
|---|---|---|
| `doc` | `spacy.tokens.Doc` | The processed Doc to search through |

Returns a list of `(match_id, start, end)` tuples:
- `match_id` — integer hash of the matched key (convert back with `nlp.vocab.strings[match_id]`)
- `start` — index of the first token of the match
- `end` — index after the last token of the match (exclusive, Python slice convention)

#### `nlp.vocab.strings[match_id]`
Converts an integer hash back to the human-readable key string.

| Parameter | Type | Description |
|---|---|---|
| `match_id` | `int` | The match ID integer returned by `matcher()` |

In [ ]:
# PhraseMatcher
COMAPNY_ALIASES = {
    "APPLE": [
          	'apple', "AAPL", 'macbook', 'imac', 'apple watch', 'steve job', 'tim cook', 'ios', 'iPhone'
    ],
    "SAMSUNG": [
        'samsung', 'samsung electronic', 'galaxy', 'galaxy AI',
    ]
}

In [ ]:
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

for company_key, aliases in COMPANY_ALIASES.items():
  patterns = [nlp.make_doc(alias) for alias in aliases]
  matcher.add(company_key, patterns)

print("Phrase Matcher:")

for key, aliases in COMPANY_ALIASES.items():
  print(f"{key}: {len(aliases)} aliases >> {aliases}")

Phrase Matcher:
APPLE: 9 aliases >> ['apple', 'AAPL', 'macbook', 'imac', 'apple watch', 'steve job', 'tim cook', 'ios', 'iPhone']
SAMSUNG: 4 aliases >> ['samsung', 'samsung electronic', 'galaxy', 'galaxy AI']


---

### 🛠️ Custom Function: `extract_company_mentions(text)`

This function is the backbone of our entity detection step. It combines NER and PhraseMatcher in a single call.

#### Purpose
Process a raw headline string and return: (1) which tracked companies are mentioned, and (2) the processed `Doc` object (reused downstream for sentiment scoring — we avoid processing the same text twice).

#### Parameters

| Parameter | Type | Description |
|---|---|---|
| `text` | `str` | A raw news headline string |

#### Returns

| Value | Type | Description |
|---|---|---|
| `companies_mentioned` | `list[str]` | Unique company keys found (from `COMPANY_ALIASES` keys). Empty list if no match. |
| `doc` | `spacy.tokens.Doc` | The full processed Doc — passed to `doc._.blob` later for sentiment |

#### Design choices
- We use a Python `set` internally to deduplicate. If both `"Apple"` and `"AAPL"` appear in one headline, we still produce one `"APPLE"` entry — not two.
- The `doc` is returned so the caller avoids re-running `nlp()` on the same text for sentiment scoring.

In [ ]:
def extract_company_mention(text):

  doc = nlp(text)
  matches = matcher(doc)

  company_mentioned = set()

  for match_id, start, end in matches:
    label = nlp.vocab.strings[match_id]
    company_mentioned.add(label)

  return list(company_mentioned), doc

In [ ]:
test_headlines = [
    "AAPL climbs after Tim Cook announces major AI partnership",
    "Samsung Galaxy S25 faces supply chain delays in Seoul",
    "Apple and Samsung both benefit from surge in semiconductor demand",
    "Fed raises interest rates — markets tumble globally",   # no match expected
    "iPhone 17 leaks point to thinner design and new camera system",
]

In [ ]:
print(f"Company Extraction Info: ")
for h in test_headlines:
  companies = extract_company_mention(h)
  print("=====" * 100)
  print(f"Headlines: {h[:70]!r}")
  print("=====" * 100)
  print(f"Aliases Matched: {companies}")

Company Extraction Info: 
Headlines: 'AAPL climbs after Tim Cook announces major AI partnership'
Aliases Matched: (['APPLE'], AAPL climbs after Tim Cook announces major AI partnership)
Headlines: 'Samsung Galaxy S25 faces supply chain delays in Seoul'
Aliases Matched: (['SAMSUNG'], Samsung Galaxy S25 faces supply chain delays in Seoul)
Headlines: 'Apple and Samsung both benefit from surge in semiconductor demand'
Aliases Matched: (['APPLE', 'SAMSUNG'], Apple and Samsung both benefit from surge in semiconductor demand)
Headlines: 'Fed raises interest rates — markets tumble globally'
Aliases Matched: ([], Fed raises interest rates — markets tumble globally)
Headlines: 'iPhone 17 leaks point to thinner design and new camera system'
Aliases Matched: (['APPLE'], iPhone 17 leaks point to thinner design and new camera system)


---
## 😊 Section 3 — Sentiment Scoring with spacytextblob

### What is Sentiment?

Think of sentiment as a **mood detector** for text. When you read a news headline your brain automatically judges whether it sounds like good news or bad news.

`spacytextblob` does this automatically. It gives us **two scores** for every headline:

---

### Score 1 — Polarity: How Positive or Negative?

```
  −1.0          −0.5          0.0          +0.5          +1.0
   │              │             │             │             │
Very           Somewhat      Neutral       Somewhat       Very
Negative       Negative                    Positive     Positive

"Samsung       "Weak chip    "Apple Q3    "Strong iPhone  "Apple
 recall"        sales"        results"     demand"       shatters
                                                         records"
```

---

### Score 2 — Subjectivity: Fact or Opinion?

```
   0.0                          0.5                          1.0
    │                            │                            │
Pure Fact                      Mixed                     Pure Opinion

"Apple revenue           "Apple delivered            "Apple is absolutely
 was $90.1 billion"       solid results"              dominating the market"
```

---

### How spacytextblob Fits Into Our spaCy Pipeline

We already have a pipeline with these steps running in order:

```
  nlp("Apple reports record revenue")
        │
        ▼
  [tokenizer]  →  splits text into words and punctuation
        │
        ▼
  [tagger]     →  labels each word: noun, verb, adjective...
        │
        ▼
  [parser]     →  finds grammar relationships between words
        │
        ▼
  [ner]        →  detects named entities: ORG, PERSON, GPE...
        │
        ▼
  [spacytextblob]  ← WE ADD THIS STEP
        │
        ▼
  doc  →  now has doc._.blob.polarity and doc._.blob.subjectivity
```

---

### 🔧 Function: `nlp.add_pipe(factory_name)`

**What it does:** Adds a new step to the end of the spaCy pipeline.

| Parameter | Type | What to pass |
|---|---|---|
| `factory_name` | `str` | The name of the component to add. We use `"spacytextblob"`. This name is registered automatically when we import `SpacyTextBlob` at the top of the notebook. |

**After calling this, every `nlp(text)` call runs sentiment scoring automatically.**

---

### 🔧 Accessing the Scores: `doc._.blob`

The `._` is spaCy's **custom attribute namespace** — a special storage area where plugins can attach their own data to a Doc.

When spacytextblob is added to the pipeline, it stores its results in that area under the name `blob`.

| What you type | What you get back |
|---|---|
| `doc._.blob.polarity` | A `float` from −1.0 to +1.0 |
| `doc._.blob.subjectivity` | A `float` from 0.0 to 1.0 |

In [ ]:
nlp.add_pipe("spacytextblob")

In [ ]:
print(f"Pipe names in the pipeline: {nlp.pipe_names}")

Pipe names in the pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner', 'spacytextblob']


---

### Let's Test It on ONE Headline First

Here is the exact flow step by step:

```
"Apple posts record iPhone sales"   ← you start with a string
               │
               ▼
          nlp(headline)              ← runs the full pipeline (including spacytextblob)
               │
               ▼
             doc                     ← a Doc object that now holds everything
               │
       ┌───────┴────────┐
       ▼                ▼
 doc._.blob.polarity    doc._.blob.subjectivity
       │                       │
  a number −1 to +1       a number 0 to 1
  (how positive?)         (fact or opinion?)
```

In [ ]:
headlines = "Apple posts record iPhone sales"

In [ ]:
from textblob.en import subjectivity
doc = nlp(headlines)

# Extracting the polarity and subjectivty from the doc

polarity = doc._.blob.polarity

subjectivity = doc._.blob.subjectivity

print(f"Headline: {headlines}")
print()
print(f"Polarity: {polarity}")
print()
print(f"Subjectivity: {subjectivity}")

Headline: Apple posts record iPhone sales

Polarity: 0.0

Subjectivity: 0.0


In [ ]:
test_headlines = [
    "Apple shatters revenue records with explosive iPhone demand",    # Very positive
    "Samsung faces massive recall after serious battery failures",     # Negative
    "Apple reports Q3 results broadly in line with expectations",      # Neutral/factual
    "Samsung shares tumble as chip sales badly disappoint analysts",   # Negative
    "Apple and Samsung both expand manufacturing in Vietnam",          # Slightly positive
    "Tim Cook confident Apple will overcome supply chain issues",      # Mild positive
]

In [ ]:
print(f"{'Headline': <55} | {'Polarity': >9} | {'Subjectivity': >12}")
print('-' * 100)

for h in test_headlines:
  doc = nlp(h)
  pol = doc._.blob.polarity
  sub = doc._.blob.subjectivity

  status = 'Positive'  if pol > 0.1 else ('Negative' if pol < 0 else 'Neutral')

  print(f"{h[:54]: <55} | {pol: >+9.3f} | {sub: >12.3f} {status}")

Headline                                                |  Polarity | Subjectivity
----------------------------------------------------------------------------------------------------
Apple shatters revenue records with explosive iPhone d  |    +0.000 |        0.000 Neutral
Samsung faces massive recall after serious battery fai  |    -0.167 |        0.833 Negative
Apple reports Q3 results broadly in line with expectat  |    +0.062 |        0.312 Neutral
Samsung shares tumble as chip sales badly disappoint a  |    -0.700 |        0.667 Negative
Apple and Samsung both expand manufacturing in Vietnam  |    +0.000 |        0.000 Neutral
Tim Cook confident Apple will overcome supply chain is  |    +0.500 |        0.833 Positive


In [ ]:
h = input("Enter your headlines: ")

doc = nlp(h)
pol = doc._.blob.polarity
sub = doc._.blob.subjectivity

status = 'Positive'  if pol > 0.1 else ('Negative' if pol < 0 else 'Neutral')

print(f"{h[:54]: <55} | {pol: >+9.3f} | {sub: >12.3f} {status}")

Enter your headlines: Apple is facing major crisis in their company due to lack of new inventions
Apple is facing major crisis in their company due to l  |    +0.025 |        0.443 Neutral
